## Load both Parquets

In [2]:
import pandas as pd

reddit = pd.read_parquet("reddit_cleaned_1min.parquet")
kalshi = pd.read_parquet("../kalshi_data/kalshi_all_games.parquet")

## Ensure all the timestamps are normalized to UTC

In [3]:
reddit["timestamp"] = pd.to_datetime(reddit["timestamp"], utc=True)
kalshi["timestamp"] = pd.to_datetime(kalshi["timestamp"], utc=True)

## Now, compute if we're UP or DOWN or FLAT

In [4]:
kalshi = kalshi.sort_values(["reddit_game", "timestamp"])
kalshi["price_next"] = kalshi.groupby("reddit_game")["price_close"].shift(-1)
kalshi["delta"] = kalshi["price_next"] - kalshi["price_close"]
kalshi["label"] = kalshi["delta"].apply(lambda d: "UP" if d > 0 else ("DOWN" if d < 0 else "FLAT"))

## Since we kept the reddit thread name in the Kalshi dataset, we can do an inner join

In [5]:
merged = reddit.merge(
    kalshi.drop(columns=["week"]),
    left_on=["game", "timestamp"],
    right_on=["reddit_game", "timestamp"],
    how="inner",
)

## Drop redundant reddit column

In [6]:
merged = merged.drop(columns=["reddit_game"])

## Sanity Check

In [9]:
print(f"Shape: {merged.shape}")
print(f"Games: {merged['game'].nunique()}")
print(f"Label distribution:\n{merged['label'].value_counts()}")
print(f"Columns: {list(merged.columns)}")
merged.head()

Shape: (51191, 22)
Games: 214
Label distribution:
label
FLAT    24634
DOWN    13357
UP      13200
Name: count, dtype: int64
Columns: ['timestamp', 'comment_count', 'avg_score', 'all_text_clean', 'comments_list', 'game_id', 'week', 'game', 'thread_url', 'offset_from_kickoff_min', 'is_pregame', 'price_close', 'price_open', 'price_mean', 'price_high', 'price_low', 'volume', 'kalshi_ticker', 'kalshi_title', 'price_next', 'delta', 'label']


,timestamp,comment_count,avg_score,all_text_clean,comments_list,game_id,week,game,thread_url,offset_from_kickoff_min,...,price_open,price_mean,price_high,price_low,volume,kalshi_ticker,kalshi_title,price_next,delta,label
0,2025-09-07 18:56:00+00:00,3.0,6.0,I’m gonna be clenched every time CMC touches t...,[I’m gonna be clenched every time CMC touches ...,week01_49ers__SF__at_Seahawks__SEA_buckets,1.0,49ers (SF) at Seahawks (SEA),https://www.reddit.com/r/nfl/comments/1nb1j1f/...,-39.0,...,0.56,0.56,0.56,0.56,1050.0,KXNFLGAME-25SEP07SFSEA-SF,San Francisco vs Seattle Winner?,0.56,0.0,FLAT
1,2025-09-07 18:57:00+00:00,1.0,8.0,CMC active - I’m totally confident he’s gonna ...,[CMC active - I’m totally confident he’s gonna...,week01_49ers__SF__at_Seahawks__SEA_buckets,1.0,49ers (SF) at Seahawks (SEA),https://www.reddit.com/r/nfl/comments/1nb1j1f/...,-38.0,...,0.56,0.56,0.56,0.56,177.0,KXNFLGAME-25SEP07SFSEA-SF,San Francisco vs Seattle Winner?,0.56,0.0,FLAT
2,2025-09-07 18:58:00+00:00,0.0,0.0,,[],week01_49ers__SF__at_Seahawks__SEA_buckets,1.0,49ers (SF) at Seahawks (SEA),https://www.reddit.com/r/nfl/comments/1nb1j1f/...,-37.0,...,0.56,0.56,0.56,0.56,302.0,KXNFLGAME-25SEP07SFSEA-SF,San Francisco vs Seattle Winner?,0.56,0.0,FLAT
3,2025-09-07 18:59:00+00:00,2.0,7.5,Love to see best buds have a friendly and resp...,[Love to see best buds have a friendly and res...,week01_49ers__SF__at_Seahawks__SEA_buckets,1.0,49ers (SF) at Seahawks (SEA),https://www.reddit.com/r/nfl/comments/1nb1j1f/...,-36.0,...,0.56,0.56,0.56,0.56,467.0,KXNFLGAME-25SEP07SFSEA-SF,San Francisco vs Seattle Winner?,0.56,0.0,FLAT
4,2025-09-07 19:00:00+00:00,2.0,8.5,Never forget that Seahawks = Toots </s> Let’s ...,"[Never forget that Seahawks = Toots, Let’s go ...",week01_49ers__SF__at_Seahawks__SEA_buckets,1.0,49ers (SF) at Seahawks (SEA),https://www.reddit.com/r/nfl/comments/1nb1j1f/...,-35.0,...,0.56,0.56,0.56,0.56,220.0,KXNFLGAME-25SEP07SFSEA-SF,San Francisco vs Seattle Winner?,0.56,0.0,FLAT


## Finally, save as parquet :)

In [8]:
merged.to_parquet("reddit_kalshi_joined.parquet", index=False)